# Manuscript figures and tables (Julia)

This notebook computes every number, table and figure of the paper from the per-run $\ln Q(N)$ in `../data/`, and
writes `data/figure_data.jld2`. The published figures are drawn from that file by `make_figures_mpl.ipynb`; the
Plots.jl versions this notebook saves to `julia/` are previews.

| Figure | Content |
|---|---|
| Fig. 1, Tables I–II | $\ln Q(N)$, $P(N)$ at coexistence, and the argon binodal against TMMC and experiment |
| Figs. 2–3 | the zeros at all ten LJ isotherms, and their classification relative to the ring |
| Fig. 4 | the square-well fluid, $\lambda = 1.5$ and $1.75$ |
| Fig. 5 | zeros of each independent run |
| Fig. 6 | thermodynamics from the zeros |
| Figs. 7–8 | Lee's expansion and where it breaks down |

**Outputs.** Tables are written to `cache/` as CSV (`table1_lnysat.csv`, `table2_densities.csv`,
`numbers_for_manuscript.csv`, `numbers_square_well.csv`). The zeros refined at 2048 bits are cached in `cache/*.jld2`;
set `FORCE = true` to recompute them. Inputs produced by earlier analysis steps are listed in `../data/README.md`.

**Conventions.**
- $z = e^{\beta\mu}$ with $\Lambda^{-3N}$ inside $Q(N)$. Literature comparisons use the activity $y = z/\Lambda^3$.
- $z_c$ is the equal-weight fugacity, $\Xi_{\rm liq}(z)=\Xi_{\rm gas}(z)$ on the real axis, with the sum split at
  the minimum of $\ln Q(N)+N\ln z$ between the two peaks.
- A zero is on the ring if $|\ln(|z|/z_c)| < 0.03$, inside it if below, and outside it if above.


## Setup

In [ ]:
import Pkg
Pkg.activate(@__DIR__; io=devnull)   # figures/Project.toml: GCWL from ".." plus Plots

using GCWL, JLD2, Printf, Statistics, Plots
gr()

const ROOT   = normpath(joinpath(@__DIR__, ".."))
const LJDIR  = joinpath(ROOT, "data", "lennard_jones")
const SWDIR  = joinpath(ROOT, "data", "square_well")
const FIGOUT = joinpath(ROOT, "figures")
const CACHE  = joinpath(FIGOUT, "cache")
mkpath(FIGOUT); mkpath(CACHE)

# ── switches ─────────────────────────────────────────────────────────────────
const FORCE  = false      # true: ignore every cache in figures/cache/ and recompute
const FIG5_T = "117.05"   # isotherm of Fig. 5 (independent runs)

setprecision(BigFloat, 2048)  # e^-600 ≈ 2^-866: dominated zeros need more than 1024 bits

In [ ]:
# ── style of the Plots.jl previews ──────────────────────────────────────────
const BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
const NAVY, INK2, GRIDC  = "#0d366b", "#52514e", "#e4e3df"
const TGRAD = cgrad(["#cde2fb", "#2a78d6", "#0d366b"])      # light → dark = cold → hot
const COLC  = Dict(:inner => AQUA, :ring => ORANGE, :outer => BLUE)
const MKC   = Dict(:inner => :utriangle, :ring => :circle, :outer => :diamond)
const CLASS_LABEL = Dict(:inner => "inside the ring", :ring => "on the ring", :outer => "outside the ring")

default(fontfamily="sans-serif", framestyle=:box, grid=true, gridcolor=GRIDC, gridalpha=1.0,
        foreground_color_axis=INK2, tickfontcolor=INK2, guidefontcolor=INK2,
        titlefontsize=10, guidefontsize=9, tickfontsize=8, legendfontsize=7,
        background_color=:white, background_color_inside=:white)

tcol(Tstar) = TGRAD[clamp((Tstar - 0.75) / 0.45, 0, 1)]

# ── journal styling, applied by save_fig ──
# Each figure is redrawn at its printed size (100 px = 1 in, aspect ratio kept), so the font sizes below are print
# sizes. No top-level titles: only the panel label "(a)" … is kept, left-aligned.
const TEXTW, COLW = 510 / 72, 246 / 72                       # revtex4-2 two-column widths, inches
const PRINT_W = Dict("fig1_validation" => TEXTW, "fig2_all_zeros" => COLW, "fig3_zero_classes" => TEXTW,
                     "fig4_square_well" => TEXTW, "fig5_independent_runs" => COLW,
                     "fig6_thermo_from_roots" => 0.85TEXTW, "fig7_lee_expansion" => TEXTW,
                     "fig8_series_breakdown" => TEXTW)
"Capitalize the first word of a label, unless it is math or a symbol such as ln."
function capfirst(s)
    s = string(s); m = match(r"^([a-z][a-z]+)\b", s)
    (m === nothing || m[1] in ("ln", "max", "min")) ? s : uppercasefirst(s)
end
function journal_style!(fig, width_in)
    w, h = fig[:size]
    plot!(fig; size = (round(Int, 100width_in), round(Int, 100width_in * h / w)))
    for (i, sp) in enumerate(fig.subplots)
        m = match(r"^\(([a-z])\)", string(sp[:title]))
        plot!(fig; subplot = i, title = m === nothing ? "" : "($(m[1]))", titlelocation = :left,
              titlefontsize = 10, guidefontsize = 9, tickfontsize = 8, legendfontsize = 7,
              xlabel = capfirst(sp[:xaxis][:guide]), ylabel = capfirst(sp[:yaxis][:guide]))
        for se in sp.series_list
            se[:label] = capfirst(se[:label])
        end
    end
    return fig
end

"Write a Plots.jl preview of fig to julia/<name>.{pdf,png} in journal style and return it for display."
function save_fig(fig, name)
    haskey(PRINT_W, name) && journal_style!(fig, PRINT_W[name])
    mkpath(joinpath(FIGOUT, "julia"))
    for ext in ("pdf", "png")
        savefig(fig, joinpath(FIGOUT, "julia", "$name.$ext"))
    end
    println("wrote figures/julia/$name.{pdf,png}")
    return fig
end

"JLD2 cache in cache/: return the stored value, or compute, store and return it."
function cached(f, name)
    path = joinpath(CACHE, name)
    isfile(path) && !FORCE && return load(path, "value")
    v = f()
    jldsave(path; value = v)
    return v
end

In [ ]:
# ── argon / LJ constants and literature tables ──────────────────────────────
const TEMPS  = ["87.79", "93.64", "99.49", "105.35", "111.20",
                "117.05", "122.90", "128.76", "134.61", "140.46"]
const TEMP_K = parse.(Float64, TEMPS)
const ε_kB_Ar, M_Ar, σ_Ar_Å = 117.05, 39.948, 3.4          # K, g/mol, Å
const V_LJ = 512.0                                          # σ³
const LJ_TO_KJKG = ε_kB_Ar * 1.380649e-23 * 6.02214076e23 / (M_Ar * 1e-3) / 1000

# SEGC-WL and experiment: Desgranges & Delhommelle, JCP 136, 184107 (2012), Tables I and II.
# Their "ln z_sat" is our ln y_sat (activity).
const REF_SEGC_LNY = [-5.684, -5.194, -4.775, -4.415, -4.104, -3.833, -3.592, -3.382, -3.193, -3.028]
# TMMC: original GC-TMMC data for the same system (V = 512σ³, r_c = 3σ + LRC), NIST SRSW,
# https://mmlapps.nist.gov/srs/LJ_PURE/sattmmc.htm (method: Errington, JCP 118, 9915 (2003)).
# TMMC values are taken directly from the NIST SRSW page.
const NIST_LNY   = [-5.68312, -5.19614, -4.77887, -4.41897, -4.10680, -3.83428, -3.59507, -3.38410, -3.19707, -3.03072]
const NIST_ρLIQ_R = [0.821911, 0.799867, 0.776854, 0.752806, 0.727724, 0.701018, 0.672152, 0.640799, 0.605204, 0.563158]
const NIST_ρVAP_R = [3.62958e-3, 6.09962e-3, 9.64024e-3, 1.45128e-2, 2.10228e-2, 2.95686e-2, 4.06547e-2, 5.50768e-2, 7.41176e-2, 1.00339e-1]
const ρ_TO_GCC = M_Ar / (6.02214076e23 * (σ_Ar_Å * 1e-8)^3)   # σ⁻³ → g/cm³
const REF_TMMC_LNY = NIST_LNY
const TMMC_ρLIQ = NIST_ρLIQ_R .* ρ_TO_GCC
const TMMC_ρVAP = NIST_ρVAP_R .* ρ_TO_GCC
const EXP_ρLIQ  = [1.389, 1.352, 1.314, 1.273, 1.231, 1.184, 1.134, 1.077, 1.012, 0.934]
const EXP_ρVAP  = [0.006, 0.010, 0.0163, 0.025, 0.036, 0.051, 0.070, 0.096, 0.131, 0.182]
# Experimental critical point (Tegeler, Span & Wagner, JPCRD 28, 779 (1999)) and Desgranges' estimate
const EXP_TC, EXP_ρC, EXP_PC = 150.687, 0.5356, 48.63

## Shared functions

Coexistence (equal weight of the gas and liquid parts of $\Xi$ on the real axis), the classification of the zeros
relative to the ring, and arbitrary-precision evaluation of $\Xi$. Everything is computed directly from $\ln Q(N)$.

In [ ]:
lse(v) = (m = maximum(v); m + log(sum(exp.(v .- m))))

"""
Maxima of g(N,u) = ln Q(N) + N u, where u = ln|z|: g has a maximum wherever the forward
difference of ln Q falls through −u. Neighbouring maxima separated by a valley shallower
than `prom` ln units are merged into one.
"""
function packet_maxima(lnQ, N, u; prom = 2.0)
    d  = diff(lnQ)
    mx = [i + 1 for i in 1:length(d)-1 if d[i] + u > 0 && d[i+1] + u <= 0]
    g  = lnQ .+ N .* u
    merged = true
    while merged && length(mx) > 1
        merged = false
        for j in 1:length(mx)-1
            lo, hi = mx[j], mx[j+1]
            if min(g[lo], g[hi]) - minimum(@view g[lo:hi]) < prom
                deleteat!(mx, g[lo] < g[hi] ? j : j + 1)
                merged = true
                break
            end
        end
    end
    return mx, g
end

"Index of the valley between the outermost packets."
valley(g, mx) = mx[1] - 1 + argmin(@view g[mx[1]:mx[end]])

"Root of an increasing function F near u0: narrow bracket first, widened only on failure."
function bracket_root(F, u0; span = 0.01, maxspan = 0.6, iters = 100)
    a, b = u0 - span, u0 + span
    while !(F(a) < 0 < F(b))
        span *= 2
        span > maxspan && error("no sign change within ±$maxspan of u0 = $u0")
        a, b = u0 - span, u0 + span
    end
    for _ in 1:iters
        m = (a + b) / 2
        F(m) > 0 ? (b = m) : (a = m)
    end
    return (a + b) / 2
end

"ln Ξ_liq − ln Ξ_gas on the real axis at u = ln z, split at the valley of g there (NaN if one packet)."
function lna_real(lnQ, N, u)
    mx, g = packet_maxima(lnQ, N, u)
    length(mx) < 2 && return NaN
    ib = valley(g, mx)
    return lse(g[ib+1:end]) - lse(g[1:ib])
end

"""
Equal-weight coexistence on the real axis, seeded by the package's equal-area bisection.
Returns z_coex, the valley split, and the cumulants of N within the gas and liquid parts.
"""
function coexistence(lnQ, N; T_star, V, μ_lo = -20.0, μ_hi = -5.0)
    u0 = find_μ_coex(lnQ, T_star; N_min = 0, μ_lo = μ_lo, μ_hi = μ_hi, tol = 1e-9) / T_star
    u  = bracket_root(x -> lna_real(lnQ, N, x), u0)
    mx, g = packet_maxima(lnQ, N, u)
    ib = valley(g, mx)
    pg = exp.(g[1:ib] .- lse(g[1:ib]));  pl = exp.(g[ib+1:end] .- lse(g[ib+1:end]))
    cum(p, n) = (m = sum(p .* n); (m, sum(p .* (n .- m) .^ 2), sum(p .* (n .- m) .^ 3)))
    κg, κl = cum(pg, N[1:ib]), cum(pl, N[ib+1:end])
    return (u = u, z = exp(u), ib = ib, Nb = N[ib],
            Ngas = κg[1], Nliq = κl[1], var_gas = κg[2], var_liq = κl[2],
            ΔN = κl[1] - κg[1],
            Δ  = ((κl[1] - κg[1]) / V, (κl[2] - κg[2]) / V, (κl[3] - κg[3]) / V))
end

"Mean particle number at real fugacity r."
function meanN(lnQ, N, r)
    g = lnQ .+ N .* log(r)
    p = exp.(g .- lse(g))
    return sum(p .* N)
end

In [ ]:
# ── classification of the zeros relative to the ring ──
# D = ln(|z|/z_c); on the ring if |D| < RING_BAND, i.e. z_c ± 3%. 0.03 holds every ring zero at every T (they
# reach |D| = 0.025) without absorbing interior zeros; 0.02 cuts the ring at T* = 1.20, 0.05 swallows interior
# zeros at T* ≥ 1.05.
const RING_BAND = 0.03
classify(D) = abs(D) < RING_BAND ? :ring : (D > 0 ? :outer : :inner)

In [ ]:
# ── arbitrary precision: Newton refinement and residuals ──
"Ξ(z) and Ξ'(z) by Horner's rule; c[k] is the coefficient of z^(k-1)."
function horner(c, z)
    p, dp = zero(z), zero(z)
    for k in length(c):-1:1
        dp = dp * z + p
        p  = p * z + c[k]
    end
    return p, dp
end

"Newton-polish a root to the working BigFloat precision."
function polish_root(c, z0; maxit = 60)
    z   = Complex{BigFloat}(z0)
    tol = BigFloat(2)^(-(precision(BigFloat) - 64))
    for _ in 1:maxit
        p, dp = horner(c, z)
        dz = p / dp
        z -= dz
        abs(dz) <= tol * abs(z) && break
    end
    return z
end

"ln of the relative residual ϱ = |Ξ(z)| / Σ Q(N)|z|^N."
log_relres(c, z) = Float64(log(abs(horner(c, z)[1])) -
                           log(sum(abs.(c) .* abs(z) .^ (0:length(c)-1))))

### The ten LJ isotherms

For each isotherm: the four-run mean $\ln Q(N)$ with the long-range correction and its 450 zeros, from
`../data/lennard_jones/output/`; the zeros refined by Newton's method at 2048 bits (cached in
`cache/polished_roots_*.jld2`); equal-weight coexistence; and the classification of the zeros. Without the cache, the
refinement of all 4500 zeros takes 1–2 min.

In [ ]:
"Per-run LRC-corrected ln Q(N) for one LJ isotherm, and the sim parameters."
function load_runs_lj(T)
    runs, sim = Vector{Float64}[], nothing
    for r in 1:4
        d = joinpath(LJDIR, T, "run$r")
        isfile(joinpath(d, "final_wl.jld2")) || continue
        wl  = load(joinpath(d, "final_wl.jld2"), "wl")
        sim = load(joinpath(d, "sim.jld2"), "sim")
        push!(runs, apply_lrc_to_logQ(collect(Float64, wl.logQ_N), sim.T_σ, sim.V_σ, sim.r_cut_σ;
                                      N_min = sim.N_min))
    end
    return runs, sim
end

"""
Assemble one system: ln Q, coexistence, polished roots over the full plane, classification.
`roots` are ComplexF64 roots of Σ exp(lnQ[k]) z^(k-1).
"""
function build_system(name, lnQ, N, roots; T_star, V, μ_lo = -20.0, μ_hi = -5.0, polish = true)
    c  = coexistence(lnQ, N; T_star = T_star, V = V, μ_lo = μ_lo, μ_hi = μ_hi)
    coef = exp.(big.(lnQ))
    # Aberth returns some real negative roots with a spurious |Im|/|z| of 1e-66 … 1e-246 and no
    # conjugate partner; snap those onto the real axis before rebuilding the full plane.
    roots = [abs(imag(x)) < 1e-30 * abs(x) ? complex(real(x), 0.0) : x for x in roots]
    up = filter(z -> imag(z) >= 0, roots)
    zbig_up = if polish
        pol = cached("polished_roots_$name.jld2") do
            zs = [polish_root(coef, z) for z in up]
            (re = string.(real.(zs)), im = string.(imag.(zs)))
        end
        Complex{BigFloat}.(parse.(BigFloat, pol.re), parse.(BigFloat, pol.im))
    else
        Complex{BigFloat}.(up)
    end
    zbig = vcat(zbig_up, conj.(filter(z -> imag(z) != 0, zbig_up)))   # full plane, conjugates added
    z    = ComplexF64.(zbig)
    D    = log.(abs.(z)) .- c.u                     # geometric offset ln(|z|/z_c)
    return (name = name, T_star = T_star, V = V, lnQ = lnQ, N = N, coex = c, coef = coef,
            zbig = zbig, z = z, D = D, cls = classify.(D))
end

"Worst relative residual (ln ϱ) of a root set against the polynomial with coefficients exp.(lnQ)."
max_lnres(lnQ, roots) = (c = exp.(big.(lnQ)); maximum(log_relres(c, Complex{BigFloat}(z)) for z in roots))

function load_lj(T)
    f = jldopen(joinpath(LJDIR, "output", "yang_lee_zeros_avg_T$T.jld2"), "r")
    lnQ, N, roots, Ts = Float64.(f["lnQ_mean"]), collect(f["N_values"]), f["roots"], f["T_star"]
    lnQ_std = Float64.(f["lnQ_std"])
    close(f)
    s = build_system("LJ_T$T", lnQ, N, roots; T_star = Ts, V = V_LJ)
    return merge(s, (T_str = T, T_K = parse(Float64, T), lnQ_std = lnQ_std, roots_stored = roots))
end

LJ = Dict(T => load_lj(T) for T in TEMPS)
SYS = [LJ[T] for T in TEMPS]

println(" T*    z_coex        ΔN      inner  ring  outer  total")
for s in SYS
    length(s.z) == length(s.N) - 1 || @warn "full-plane reconstruction has $(length(s.z)) zeros, expected $(length(s.N)-1)"
    @printf(" %.2f  %.5e  %6.1f    %3d   %3d   %3d    %3d\n", s.T_star, s.coex.z,
            s.coex.ΔN, count(==(:inner), s.cls), count(==(:ring), s.cls), count(==(:outer), s.cls),
            length(s.z))
end

### The square-well state points

$\lambda = 1.5$ and $1.75$ at $V=729\sigma^3$, four runs each. There is no long-range correction (the potential is zero
beyond $\lambda\sigma$), $\ln Q(0)$ is re-anchored to 0, and $\Lambda_\sigma=1$, so $y=z$. The zeros are those in
`../data/square_well/output/`; they are checked against the four-run mean $\ln Q$ and re-solved if their residuals
are too large.

In [ ]:
const V_SW = 729.0
const SW_LAMBDAS = (1.5, 1.75)
function load_sw()
    out = []
    for f in sort(filter(x -> endswith(x, "_avg.jld2"), readdir(joinpath(SWDIR, "output"))))
        m = match(r"sw_zeros_lam([\d.]+)_T([\d.]+)_avg", f)
        λ, T = parse(Float64, m[1]), parse(Float64, m[2])
        λ in SW_LAMBDAS || continue
        dir = joinpath(SWDIR, @sprintf("lambda_%.2f", λ), @sprintf("T_%.3f", T))
        runs = Vector{Float64}[]
        for r in 1:4
            p = joinpath(dir, "run$r", "final_wl.jld2"); isfile(p) || continue
            lq = collect(Float64, load_wanglandau_jld2(p).logQ_N); lq .-= lq[1]; push!(runs, lq)
        end
        lnQ = mean(runs); N = collect(0:length(lnQ)-1)
        name  = @sprintf("SW_lam%.2f_T%.3f", λ, T)
        roots = load(joinpath(SWDIR, "output", f), "roots")
        res   = max_lnres(lnQ, roots)
        if res > -20
            # The stored zeros are not zeros of this mean ln Q: re-solve at 1024 bits and cache them in
            # cache/, leaving square_well/output alone.
            @warn @sprintf("%s: stored zeros have max ln ϱ = %.1f against the mean ln Q — re-solving", name, res)
            roots = cached("sw_roots_$name.jld2") do
                yang_lee_zeros(lnQ, 0:length(lnQ)-1; prec_bits = 1024, tol = 1e-10).roots
            end
            res = max_lnres(lnQ, roots)
        end
        s = build_system(name, lnQ, N, roots; T_star = T, V = V_SW,
                         μ_lo = -30.0, μ_hi = 5.0, polish = false)
        # equal weight per run, seeded at the mean's z_c: the package equal-area bisection lands on a
        # wrong root for two runs at λ = 1.5, T* = 1.15 (sd 0.03 instead of 5e-4)
        lnz_runs = [bracket_root(x -> lna_real(lq, N, x), s.coex.u) for lq in runs]
        push!(out, merge(s, (λ = λ, lnz_sd = std(lnz_runs), nruns = length(runs), lnQ1_dev = lnQ[2] - log(V_SW),
                             lnres = res)))
    end
    return out
end
SWS = load_sw()
LAMS = sort(unique(getfield.(SWS, :λ)))

println(" λ     T*     runs  N_max  max ln ϱ  ln Q(1) − ln V   ΔN      ring  inner  outer  ln z_coex ± sd")
for s in SWS
    @printf(" %.2f  %.3f   %d    %3d   %6.1f    %+.1e       %6.1f   %3d   %3d   %3d    %.4f ± %.4f\n", s.λ, s.T_star, s.nruns,
            last(s.N), s.lnres, s.lnQ1_dev, s.coex.ΔN, count(==(:ring), s.cls), count(==(:inner), s.cls), count(==(:outer), s.cls),
            s.coex.u, s.lnz_sd)
end

## Fig. 1 and Tables I–II: validation against argon

(a) $\ln Q(N)$ at the ten isotherms; (b) $P(N)$ at coexistence; (c) the coexistence densities against TMMC and
experiment, with the critical point from a scaling fit. Tables I and II are written to `cache/`.

In [ ]:
# ── per-run coexistence, following Desgranges: bisect each run, std over runs ──
function coexistence_table(T)
    runs, sim = load_runs_lj(T)
    μs  = [find_μ_coex(lq, sim.T_σ; N_min = 0, μ_lo = -20.0, μ_hi = -5.0, tol = 1e-7) for lq in runs]
    den = [compute_phase_densities(lq, μ, sim.T_σ, sim.V_σ; N_min = 0) for (lq, μ) in zip(runs, μs)]
    ρl  = [ljdens_to_gcm3(d[1]; σ_Å = σ_Ar_Å, M_g_per_mol = M_Ar) for d in den]
    ρv  = [ljdens_to_gcm3(d[2]; σ_Å = σ_Ar_Å, M_g_per_mol = M_Ar) for d in den]
    # ln Ξ from the vapour half only (A_vap = A_liq = 1/2 at coexistence)
    P = map(zip(runs, μs)) do (lq, μ)
        pN = compute_pN(lq, μ, sim.T_σ; N_min = 0); nb = find_Nb_idx(pN) - 1
        lnΞ = log(2.0) + logsumexp([lq[n+1] + n * μ / sim.T_σ for n in 0:nb])
        compute_Psat_bar(lnΞ, sim.T_σ, sim.V_σ; ε_kB = ε_kB_Ar, σ_Å = σ_Ar_Å)
    end
    lny = compute_ln_y_sat.(μs, sim.T_σ, sim.Λ_σ)
    lnQ1 = [lq[2] for lq in runs]
    return (T_K = parse(Float64, T), n = length(runs), Λ = sim.Λ_σ,
            lny = mean(lny), lny_sd = std(lny), μ = mean(μs) * LJ_TO_KJKG, μ_sd = std(μs) * LJ_TO_KJKG,
            ρl = mean(ρl), ρl_sd = std(ρl), ρv = mean(ρv), ρv_sd = std(ρv), P = mean(P), P_sd = std(P),
            lnQ1_dev = mean(lnQ1) - log(sim.V_σ / sim.Λ_σ^3), lnQ1_sd = std(lnQ1))
end

TAB = [coexistence_table(T) for T in TEMPS]

println("TABLE 1 — ln y_sat (activity y = z/Λ³), mean ± sd over 4 independent runs")
println(" T (K)    μ_coex (kJ/kg)      ln y_sat GC-WL      SEGC-WL   TMMC     Δ(GC-WL − TMMC)")
for (i, t) in enumerate(TAB)
    @printf(" %6.2f   %8.2f ± %.2f    %7.3f ± %.4f     %7.3f   %7.3f    %+.3f\n", t.T_K, t.μ, t.μ_sd,
            t.lny, t.lny_sd, REF_SEGC_LNY[i], REF_TMMC_LNY[i], t.lny - REF_TMMC_LNY[i])
end
println("\nTABLE 2 — coexistence densities (g/cm³)")
println(" T (K)   ρ_liq GC-WL        exp     TMMC  |  ρ_vap GC-WL          exp     TMMC")
for (i, t) in enumerate(TAB)
    @printf(" %6.2f  %.3f ± %.3f   %.3f   %.3f  |  %.4f ± %.4f   %.4f   %.4f\n", t.T_K, t.ρl, t.ρl_sd,
            EXP_ρLIQ[i], TMMC_ρLIQ[i], t.ρv, t.ρv_sd, EXP_ρVAP[i], TMMC_ρVAP[i])
end
println("\nln Q(1) − ln(V/Λ³): the one exact value never imposed during sampling")
for t in TAB
    @printf(" %6.2f  %+.1e  (run-to-run sd %.1e)\n", t.T_K, t.lnQ1_dev, t.lnQ1_sd)
end

open(joinpath(CACHE, "table1_lnysat.csv"), "w") do io
    println(io, "T_K,mu_kJkg,mu_sd,lny_sat,lny_sd,segc,tmmc")
    for (i, t) in enumerate(TAB)
        println(io, join((t.T_K, t.μ, t.μ_sd, t.lny, t.lny_sd, REF_SEGC_LNY[i], REF_TMMC_LNY[i]), ","))
    end
end
open(joinpath(CACHE, "table2_densities.csv"), "w") do io
    println(io, "T_K,rho_liq,rho_liq_sd,rho_vap,rho_vap_sd,P_bar,P_sd,exp_rho_liq,exp_rho_vap,tmmc_rho_liq,tmmc_rho_vap")
    for (i, t) in enumerate(TAB)
        println(io, join((t.T_K, t.ρl, t.ρl_sd, t.ρv, t.ρv_sd, t.P, t.P_sd,
                          EXP_ρLIQ[i], EXP_ρVAP[i], TMMC_ρLIQ[i], TMMC_ρVAP[i]), ","))
    end
end
println("\nwrote cache/table1_lnysat.csv, cache/table2_densities.csv")

In [ ]:
# ── critical point: scaling with the first Wegner correction + rectilinear diameter ──
# ρ_l − ρ_v = C1 t^β + C2 t^(β+Δ), t = 1 − T/T_c, β = 0.326, Δ = 0.51 (the form of Singh, Kofke & Errington,
# JCP 119, 3405 (2003)); then (ρ_l + ρ_v)/2 = ρ_c + C3 (T − T_c). Units of T and ρ are those passed in.
function estimate_critical_point(T, ρl, ρv; β = 0.326, Δ = 0.51)
    d = ρl .- ρv; best = (err = Inf, Tc = NaN)
    for Tc in maximum(T) * 1.0005 : maximum(T) * 2e-5 : maximum(T) * 1.6
        t = 1 .- T ./ Tc; A = hcat(t .^ β, t .^ (β + Δ))
        err = sum((A * (A \ d) .- d) .^ 2)
        err < best.err && (best = (err = err, Tc = Tc))
    end
    B = hcat(ones(length(T)), T .- best.Tc)
    return (Tc = best.Tc, ρc = (B \ ((ρl .+ ρv) ./ 2))[1])
end
CP = estimate_critical_point(TEMP_K, getfield.(TAB, :ρl), getfield.(TAB, :ρv))
@printf("critical point estimate: T_c = %.1f K, ρ_c = %.3f g/cm³   (experiment %.2f K, %.3f g/cm³; Desgranges 151±4 K, 0.528±0.008)\n",
        CP.Tc, CP.ρc, EXP_TC, EXP_ρC)

# ── (a) ln Q(N) ──────────────────────────────────────────────────────────────
pa = plot(xlabel = "N", ylabel = "\$\\ln Q(N,V,T)\$", title = "(a) canonical partition functions",
          legend = :topleft)
for s in SYS
    plot!(pa, s.N, s.lnQ, lc = tcol(s.T_star), lw = 1.6, label = @sprintf("\$T^* = %.2f\$", s.T_star))
end

# ── (b) P(N) at coexistence ──────────────────────────────────────────────────
pb = plot(xlabel = "N", ylabel = "\$P(N)\$ at \$z_c\$", title = "(b) particle-number distribution at coexistence",
          legend = false)                       # temperatures as in (a)
for s in SYS
    g = s.lnQ .+ s.N .* s.coex.u
    plot!(pb, s.N, exp.(g .- lse(g)), lc = tcol(s.T_star), lw = 1.6, label = @sprintf("\$T^* = %.2f\$", s.T_star))
end

# ── (c) binodal against TMMC and experiment ──────────────────────────────────
pc = plot(xlabel = "ρ (g/cm³)", ylabel = "T (K)", title = "(c) vapour–liquid coexistence of argon",
          legend = :bottom, xlims = (0, 1.5))
scatter!(pc, EXP_ρLIQ, TEMP_K, m = :square, mc = :white, msc = INK2, ms = 4, label = "experiment")
scatter!(pc, EXP_ρVAP, TEMP_K, m = :square, mc = :white, msc = INK2, ms = 4, label = "")
scatter!(pc, getfield.(TAB, :ρl), TEMP_K, xerror = getfield.(TAB, :ρl_sd), mc = BLUE, msw = 0.5, ms = 3.5,
         label = "This work")
scatter!(pc, getfield.(TAB, :ρv), TEMP_K, xerror = getfield.(TAB, :ρv_sd), mc = BLUE, msw = 0.5, ms = 3.5, label = "")
# TMMC drawn after ours, smaller, so it stays visible where the two agree
scatter!(pc, TMMC_ρLIQ, TEMP_K, m = :diamond, mc = AQUA, msw = 0, ms = 2.5, label = "TMMC")
scatter!(pc, TMMC_ρVAP, TEMP_K, m = :diamond, mc = AQUA, msw = 0, ms = 2.5, label = "")
scatter!(pc, [CP.ρc], [CP.Tc], m = :star5, mc = ORANGE, msw = 0, ms = 6, label = "")   # stars described in the caption
scatter!(pc, [EXP_ρC], [EXP_TC], m = :star5, mc = :white, msc = INK2, ms = 6, label = "")

fig1 = plot(pa, pb, pc, layout = (1, 3), size = (1650, 480), dpi = 200,
            left_margin = 8Plots.mm, bottom_margin = 7Plots.mm)
save_fig(fig1, "fig1_validation")

## Figs. 2 and 3: the zeros of the LJ fluid

Fig. 2: all zeros at the ten isotherms in the complex $z$ plane. Fig. 3: (a) the number of zeros on, inside and
outside the ring at each isotherm; (b), (c) the zeros at $T^*=0.75$ and $1.20$ in units of $z_c$, so that the
equal-weight circle is the unit circle.

In [ ]:
function map_panel(s; title = "", lim = 2.0)
    w   = s.z ./ s.coex.z
    p = plot(aspect_ratio = :equal, xlims = (-lim, lim), ylims = (-lim, lim), legend = false,   # classes and colors as in (a)
             xlabel = "\$\\mathrm{Re}\\,z / z_c\$", ylabel = "\$\\mathrm{Im}\\,z / z_c\$", title = title)
    φc = range(0, 2π, length = 400)
    plot!(p, cos.(φc), sin.(φc), lc = INK2, ls = :dash, lw = 0.8, label = "")
    for cl in (:outer, :inner, :ring)
        sel = s.cls .== cl
        any(sel) || continue
        scatter!(p, real.(w[sel]), imag.(w[sel]), mc = COLC[cl], m = MKC[cl], ms = 2.8, msw = 0,
                 label = @sprintf("%s (%d)", CLASS_LABEL[cl], count(sel)))
    end
    nout = count(x -> abs(real(x)) > lim || abs(imag(x)) > lim, w)
    nout > 0 && annotate!(p, 0.95lim, -0.85lim, text("$nout beyond the frame", 7, INK2, :right))
    return p
end

"Leading ring zero z_0: the ring zero with the smallest positive argument."
function leading_ring_zero(s)
    idx = findall(i -> s.cls[i] === :ring && imag(s.z[i]) > 0, eachindex(s.z))
    return s.z[idx[argmin(angle.(s.z[idx]))]]
end

s75, s120 = LJ["87.79"], LJ["140.46"]

# Fig. 2: every zero at all ten isotherms, full plane, raw fugacity
zmax = maximum(s.coex.z for s in SYS)
const ZU = 1e-5   # Fig. 2 axes in units of 10⁻⁵
pa = plot(xlabel = "\$\\mathrm{Re}\\,z\\;(10^{-5})\$", ylabel = "\$\\mathrm{Im}\\,z\\;(10^{-5})\$", aspect_ratio = :equal,
          legend = :outerright, legend_title = "\$T^*\$", xlims = (-1.3zmax, 1.3zmax) ./ ZU, ylims = (-1.3zmax, 1.3zmax) ./ ZU)
for s in SYS
    scatter!(pa, real.(s.z) ./ ZU, imag.(s.z) ./ ZU, mc = tcol(s.T_star), ms = 1.6, msw = 0, alpha = 0.85,
             label = @sprintf("%.2f", s.T_star))
end
pc = map_panel(s75;  title = "(b) \$T^* = 0.75\$")
pd = map_panel(s120; title = "(c) \$T^* = 1.20\$")

# leading ring zero vs Lee's first order, θ0 = π/ΔN (numbers for the text)
z1s = [leading_ring_zero(s) for s in SYS]

# Fig. 3(a): all zeros at each isotherm, stacked: on the ring (bottom), inside, outside (overlaid cumulative bars)
Ts_ = getfield.(SYS, :T_star)
nr = [count(==(:ring), s.cls) for s in SYS]; ni = [count(==(:inner), s.cls) for s in SYS]; no = [count(==(:outer), s.cls) for s in SYS]
pb = plot(xlabel = "\$T^*\$", ylabel = "number of zeros", title = "(a) where the zeros are", legend = :top,
          ylims = (0, 640), xticks = 0.8:0.1:1.2)
bar!(pb, Ts_, nr .+ ni .+ no, bar_width = 0.035, fc = COLC[:outer], lw = 0, label = CLASS_LABEL[:outer])
bar!(pb, Ts_, nr .+ ni, bar_width = 0.035, fc = COLC[:inner], lw = 0, label = CLASS_LABEL[:inner])
bar!(pb, Ts_, nr, bar_width = 0.035, fc = COLC[:ring], lw = 0, label = CLASS_LABEL[:ring])

fig2 = plot(pa, size = (900, 760), dpi = 200, left_margin = 6Plots.mm, bottom_margin = 6Plots.mm)
save_fig(fig2, "fig2_all_zeros")
fig3 = plot(pb, pc, pd, layout = (1, 3), size = (2100, 680), dpi = 200,
            left_margin = 14Plots.mm, bottom_margin = 8Plots.mm)
save_fig(fig3, "fig3_zero_classes")

println(" T*    Im z1/z_c     π/ΔN          ratio")
for (s, z1) in zip(SYS, z1s)
    @printf(" %.2f  %.6f     %.6f      %.5f\n", s.T_star, imag(z1) / s.coex.z, π / s.coex.ΔN, imag(z1) / s.coex.z / (π / s.coex.ΔN))
end

println(" T*    Re z1         Im z1         inside  on ring  outside")
for (s, z1) in zip(SYS, z1s)
    @printf(" %.2f  %.4e   %.4e    %3d     %3d      %3d\n", s.T_star, real(z1), imag(z1),
            count(==(:inner), s.cls), count(==(:ring), s.cls), count(==(:outer), s.cls))
end

### Population, spacing and leading zero of the ring

Numbers quoted in the text: the number of ring zeros against $\Delta N$, their median angular spacing against
$2\pi/\Delta N$, the distance of the leading zero $z_0$ from $|z|=z_c$, and the widest excursion of the ring from
$|z|=z_c$.

In [ ]:
ring_count = [count(==(:ring), s.cls) for s in SYS]
spacing = map(SYS) do s
    θs = sort([angle(x) for (x, c) in zip(s.z, s.cls) if c === :ring && imag(x) > 0])
    median(diff(θs))
end
lead_w = [abs(log(abs(z1)) - s.coex.u) for (s, z1) in zip(SYS, z1s)]
ring_D = [maximum(abs.(s.D[s.cls .=== :ring])) for s in SYS]   # widest excursion of the ring from |z| = z_c

println(" T*    ring   ΔN      ring/ΔN  |  Δθ (median)  2π/ΔN     ratio  |  |ln(|z0|/z_c)|   max |ln(|z|/z_c)| on the ring")
for (i, s) in enumerate(SYS)
    @printf(" %.2f   %4d  %6.1f   %.3f    |  %.6f      %.6f  %.3f  |  %.2e          %.4f\n", s.T_star, ring_count[i], s.coex.ΔN,
            ring_count[i] / s.coex.ΔN, spacing[i], 2π / s.coex.ΔN, spacing[i] / (2π / s.coex.ΔN), lead_w[i], ring_D[i])
end

## Figs. 5 and 6: independent runs, and thermodynamics from the zeros

Fig. 5: the zeros of each of the four runs at $T^*=1.00$ (`FIG5_T`), over those of the four-run mean. Fig. 6: (a) the
density and (b) the isothermal compressibility from the zeros, against the same quantities computed directly from
$\ln Q$. The next cell gives the run-to-run spread of the leading zero at $T^*=0.75$, $1.00$ and $1.20$. The zeros of
each run are read from `../data/lennard_jones/analysis/per_run_roots.jld2`; without that file they are computed
(Aberth–Ehrlich at 1024 bits, about 40 s per run) and cached in `cache/`.

In [ ]:
const PER_RUN = joinpath(LJDIR, "analysis", "per_run_roots.jld2")
const PER_RUN_DICT = isfile(PER_RUN) ? load(PER_RUN, "roots") : Dict()

"Zeros of one run's ln Q: from per_run_roots.jld2 if there, else Aberth at 1024 bits (cached in cache/)."
function run_roots(T, r)
    haskey(PER_RUN_DICT, (T, r)) && !FORCE && return PER_RUN_DICT[(T, r)]
    cached("run_roots_T$(T)_run$r.jld2") do
        runs, _ = load_runs_lj(T)
        yang_lee_zeros(runs[r], 0:length(runs[r])-1; prec_bits = 1024, tol = 1e-10).roots
    end
end

s3 = LJ[FIG5_T]; zc3 = s3.coex.z
RR = [run_roots(FIG5_T, r) for r in 1:4]
RUNC = [ORANGE, "#56b4e9", AQUA, NAVY]

pa = plot(aspect_ratio = :equal, xlabel = "\$\\mathrm{Re}\\,z / z_c\$", ylabel = "\$\\mathrm{Im}\\,z / z_c\$", legend = :topright,
          title = @sprintf("four independent runs, \$T^* = %.2f\$", s3.T_star), xlims = (-2.2, 2.2), ylims = (-2.2, 2.2))
scatter!(pa, real.(s3.z) ./ zc3, imag.(s3.z) ./ zc3, mc = "#bdbcb7", ms = 4, msw = 0, label = "averaged ln Q")
for r in 1:4
    scatter!(pa, real.(RR[r]) ./ zc3, imag.(RR[r]) ./ zc3, mc = RUNC[r], ms = 1.8, msw = 0, label = "run $r")
end

"The n leading zeros (smallest positive argument); used for the run-to-run spread quoted in the text."
lead(zs; n = 5) = sort(filter(z -> real(z) > 0 && imag(z) > 0, zs), by = angle)[1:n]

# coarse grid over ±1 plus a dense grid across the coexistence peak of Var N (width ~ 1/ΔN in βμ)
βμ  = sort(vcat(range(s3.coex.u - 1.0, s3.coex.u + 1.0, length = 400), range(s3.coex.u - 0.03, s3.coex.u + 0.03, length = 600)))
βμm = sort(vcat(range(s3.coex.u - 1.0, s3.coex.u + 1.0, length = 40), range(s3.coex.u - 0.02, s3.coex.u + 0.02, length = 25)))
ρ_direct = [meanN(s3.lnQ, s3.N, exp(u)) / s3.V for u in βμ]
varN(u) = (g = s3.lnQ .+ s3.N .* u; p = exp.(g .- lse(g)); m = sum(p .* s3.N); sum(p .* (s3.N .- m) .^ 2))
var_direct = varN.(βμ)
ρ_roots   = [density_from_roots(s3.roots_stored, exp(u), s3.V, 0) for u in βμm]
var_roots = [var_N_from_roots(s3.roots_stored, exp(u), 0) for u in βμm]

pb = plot(xlabel = "\$\\beta\\mu - \\beta\\mu_c\$", ylabel = "\$\\rho\\;(\\sigma^{-3})\$", legend = :bottomright, title = "(a) density from the roots")
plot!(pb, βμ .- s3.coex.u, ρ_direct, lc = INK2, lw = 2, label = "from ln Q")
scatter!(pb, βμm .- s3.coex.u, ρ_roots, mc = ORANGE, ms = 3.5, msw = 0, label = "\$\\rho = V^{-1}\\sum_k z/(z - z_k)\$")
# (b) isothermal compressibility, κ_T = V Var N / (k_B T ⟨N⟩²) in σ³/ε (k_B T = T*), from the roots and from ln Q
κT(v, ρ) = v / (s3.T_star * s3.V * ρ^2)
κ_direct = κT.(var_direct, ρ_direct)
κ_roots  = κT.(var_roots, ρ_roots)
pc = plot(xlabel = "\$\\beta\\mu - \\beta\\mu_c\$", ylabel = "\$\\kappa_T\\;(\\sigma^3/\\epsilon)\$", yscale = :log10, legend = :topleft,
          title = "(b) isothermal compressibility from the roots")
plot!(pc, βμ .- s3.coex.u, κ_direct, lc = INK2, lw = 2, label = "from ln Q")
scatter!(pc, βμm .- s3.coex.u, κ_roots, mc = ORANGE, ms = 3.5, msw = 0, label = "from the roots")
fig5 = plot(pa, size = (900, 820), dpi = 200, left_margin = 6Plots.mm, bottom_margin = 6Plots.mm)
save_fig(fig5, "fig5_independent_runs")
fig6 = plot(pb, pc, layout = (1, 2), size = (1700, 620), dpi = 200,
            left_margin = 8Plots.mm, bottom_margin = 6Plots.mm)
save_fig(fig6, "fig6_thermo_from_roots")

@printf("Var N peak: from ln Q %.4g at βμ−βμ_c = %+.4f; from roots %.4g at %+.4f\n",
        maximum(var_direct), βμ[argmax(var_direct)] - s3.coex.u, maximum(var_roots), βμm[argmax(var_roots)] - s3.coex.u)
@printf("max relative |Var_roots − Var_lnQ| = %.1e\n", maximum(abs(v - varN(u)) / varN(u) for (v, u) in zip(var_roots, βμm)))
@printf("κ_T peak (σ³/ε): from ln Q %.4g; max relative |κ_roots − κ_lnQ| = %.1e\n", maximum(κ_direct),
        maximum(abs(κr - κT(varN(u), meanN(s3.lnQ, s3.N, exp(u)) / s3.V)) / κT(varN(u), meanN(s3.lnQ, s3.N, exp(u)) / s3.V) for (κr, u) in zip(κ_roots, βμm)))
@printf("max |ρ_roots − ρ_lnQ| = %.1e 1/σ³\n",
        maximum(abs(ρr - meanN(s3.lnQ, s3.N, exp(u)) / s3.V) for (ρr, u) in zip(ρ_roots, βμm)))

In [ ]:
# ── run-to-run spread of the leading zero at the isotherms with per-run zeros ──
println(" T*    leading zero, 4 runs:  sd(Re z1)/Re z1   sd(Im z1)/Im z1   |z1(avg) − mean z1(runs)| / |z1|")
for T in ("87.79", "117.05", "140.46")
    s = LJ[T]
    z1r = [lead(run_roots(T, r); n = 1)[1] for r in 1:4]
    z1a = leading_ring_zero(s)
    @printf(" %.2f              %.1e            %.1e               %.1e\n", s.T_star,
            std(real.(z1r)) / real(z1a), std(imag.(z1r)) / imag(z1a), abs(z1a - mean(z1r)) / abs(z1a))
end

## Fig. 7: Lee's expansion

The zeros of the four-run mean $\ln Q$ against the first-order circle and the second-order bent ring of Lee's
expansion, anchored at the equal-weight $z_c$. The predictions are matched to the ring zeros in the upper half plane,
numbered $k=0,1,\ldots$ by increasing argument.

In [ ]:
"Lee's expansion to 1st/2nd/3rd order, t_k = Σ (g_n/n!) w_k^n with w_k = η_k/V, and the matched exact ring zeros."
function lee_predictions(s; nz = 30)
    c = s.coex; V = s.V; Δ1, Δ2, Δ3 = c.Δ
    g1 = 1 / Δ1; g2 = -Δ2 / Δ1^3; g3 = 3Δ2^2 / Δ1^5 - Δ3 / Δ1^4
    lna = lna_real(s.lnQ, s.N, c.u)                     # ≈ 0 by construction
    zp = [ComplexF64[] for _ in 1:3]
    for k in 0:nz-1
        w  = (2π * im * (k + 0.5) - lna) / V
        h1 = g1 * w; h2 = h1 + g2 / 2 * w^2; h3 = h2 + g3 / 6 * w^3
        push!(zp[1], c.z * exp(h1)); push!(zp[2], c.z * exp(h2)); push!(zp[3], c.z * exp(h3))
    end
    exact = sort([x for (x, cl) in zip(s.z, s.cls) if cl === :ring && imag(x) > 0], by = angle)
    return (z1 = zp[1], z2 = zp[2], z3 = zp[3], exact = exact, Δ = c.Δ, lna = lna, zc = c.z)
end

"Lee's second-order bend coefficient, ΔVar/ΔN."
bend_pred(s) = (s.coex.var_liq - s.coex.var_gas) / s.coex.ΔN

LEE  = Dict(s.T_str => lee_predictions(s) for s in SYS)
BEND = Dict(s.name => bend_pred(s) for s in vcat(SYS, SWS))

# (a)–(c): the whole plane at three isotherms, every exact zero against Lee's expansion at 1st/2nd/3rd order,
# continued around the whole ring (k = 0 … number of ring zeros in the upper half; conjugates added)
const LEE_ORD = ((:z1, BLUE, :diamond, "1st order (\$\\Delta_1\$)"), (:z2, ORANGE, :utriangle, "+ 2nd order (\$\\Delta_2\$)"),
                 (:z3, "#c0392b", :star5, "+ 3rd order (\$\\Delta_3\$)"))
function lee_full_panel(s, tag; lim = 1.5)
    nring = count(i -> s.cls[i] === :ring && imag(s.z[i]) > 0, eachindex(s.z))
    r = lee_predictions(s; nz = nring)
    p = plot(aspect_ratio = :equal, xlims = (-lim, lim), ylims = (-lim, lim), legend = tag == "a" ? :bottomleft : false, xticks = -1:1:1, yticks = -1:1:1,
             xlabel = "\$\\mathrm{Re}\\,z / z_c\$", ylabel = "\$\\mathrm{Im}\\,z / z_c\$", title = @sprintf("(%s) \$T^* = %.2f\$", tag, s.T_star))
    w = s.z ./ r.zc
    scatter!(p, real.(w), imag.(w), mc = :black, ms = 2.2, msw = 0, label = "exact zeros")
    for (f, col, m, lab) in LEE_ORD
        zs = getfield(r, f) ./ r.zc
        zs = vcat(zs, conj.(zs)); zs = filter(z -> isfinite(z) && abs(z) < 2lim, zs)
        scatter!(p, real.(zs), imag.(zs), mc = col, m = m, ms = 2.0, msw = 0, alpha = 0.8, label = lab)
    end
    return p
end
# (d)–(f): the same three isotherms zoomed in on the first NZOOM zeros above the positive real axis
const NZOOM = 25
function lee_zoom_panel(s, tag)
    r = lee_predictions(s; nz = NZOOM); zc = r.zc
    pts = vcat(r.exact[1:min(NZOOM, length(r.exact))], r.z1, r.z2, r.z3) ./ zc
    xr = extrema(real.(pts)); yr = (0.0, maximum(imag.(pts)))
    pad = 0.06 * (yr[2] - yr[1])
    lo, hi = xr[1] - pad, xr[2] + pad
    st = 0.2; for d in (0.01, 0.02, 0.05, 0.1, 0.2); (hi - lo) / d <= 3.5 && (st = d; break); end   # ≤ 4 ticks
    p = plot(xlims = (lo, hi), ylims = (yr[1] - pad, yr[2] + pad), legend = false, xticks = ceil(lo / st) * st : st : hi,
             xlabel = "\$\\mathrm{Re}\\,z / z_c\$", ylabel = "\$\\mathrm{Im}\\,z / z_c\$", title = @sprintf("(%s) \$T^* = %.2f\$, first %d zeros", tag, s.T_star, NZOOM))
    w = s.z ./ zc
    scatter!(p, real.(w), imag.(w), mc = :black, ms = 4, msw = 0)
    for (f, col, m, lab) in LEE_ORD
        zs = getfield(r, f) ./ zc
        scatter!(p, real.(zs), imag.(zs), mc = col, m = m, ms = 3.4, msw = 0, alpha = 0.85)
    end
    return p
end
pA = lee_full_panel(LJ["87.79"], "a"); pB = lee_full_panel(LJ["134.61"], "b"); pC = lee_full_panel(LJ["140.46"], "c")
pD = lee_zoom_panel(LJ["87.79"], "d"); pE = lee_zoom_panel(LJ["134.61"], "e"); pF = lee_zoom_panel(LJ["140.46"], "f")
fig7 = plot(pA, pB, pC, pD, pE, pF, layout = (2, 3), size = (2100, 1450), dpi = 200, left_margin = 6Plots.mm, bottom_margin = 6Plots.mm)
save_fig(fig7, "fig7_lee_expansion")

swcol(λ) = Dict(1.5 => BLUE, 1.75 => ORANGE)[λ]

println(" system            Δ1       Δ2       Δ3        ln a at z_c   ΔVar/ΔN")
for s in vcat(SYS, SWS)
    @printf(" %-16s  %.4f  %+.4f  %+.4f     %+.1e     %.5f\n", s.name, s.coex.Δ...,
            lna_real(s.lnQ, s.N, s.coex.u), BEND[s.name])
end

## Fig. 8: reach of the circle, term sizes, and density from Lee's zeros

At second order, zero $k$ is displaced off the circle by
$$\frac{\text{drift}}{\text{spacing}}=\pi\left(k+\tfrac12\right)^2\frac{\Delta\mathrm{Var}}{\Delta N^2},$$
so the circle locates the zeros until this reaches half a spacing:
$$\varepsilon_1+\tfrac12=\frac{1}{\sqrt{2\pi}}\,\frac{\Delta N}{\sqrt{|\Delta\mathrm{Var}|}}.$$
The observed reach $k_1^{\rm obs}$ is the first $k$ at which the first-order prediction misses its zero by half a
spacing or more, $e_k=|t_k^{\rm exact}-w_k/\Delta_1|/(2\pi/\Delta N)\ge\tfrac12$, on the four-run mean $\ln Q$. The rows
are read from `../data/lennard_jones/analysis/lee_expansion_lj.jld2` and `../data/square_well/output/lee_expansion_sw.jld2`,
which also give the errors of the second and third orders quoted with Fig. 7.
Colour is $T/T_c$, with $T_c$ from the scaling fit of Fig. 1 (`estimate_critical_point`).

In [ ]:
# ── Fig. 8: reach of the circle, term sizes, and density from Lee's zeros ──
const LJ_LEE = joinpath(LJDIR, "analysis", "lee_expansion_lj.jld2")
const SW_LEE = joinpath(SWDIR, "output", "lee_expansion_sw.jld2")
isfile(LJ_LEE) || Base.include(Module(:LJM), joinpath(LJDIR, "analysis", "lee_expansion_lj.jl"))
isfile(SW_LEE) || Base.include(Module(:SWM), joinpath(SWDIR, "lee_expansion_sw.jl"))
ljrows = load(LJ_LEE, "rows"); swrows = [r for r in load(SW_LEE, "rows") if r.λ in SW_LAMBDAS]

# Errors of Lee's expansion on the ring zeros, in zero spacings (numbers in the text with Fig. 7)
firstworse(e) = (i = findfirst(k -> e[k, 3] > e[k, 2], 1:size(e, 1)); i === nothing ? "–" : string(i - 1))
println(" system          T*      error / spacing at k = 10, orders 1 / 2 / 3     first k at which order 3 is further than order 2")
for r in vcat(ljrows, swrows)
    @printf(" %-14s  %.3f   %.2g / %.2g / %.2g                      %s\n", r.λ == 0 ? "LJ" : @sprintf("SW λ=%.2f", r.λ),
            r.T, r.err[11, 1:3]..., firstworse(r.err))
end

# T_c from our own coexistence densities, same fit as Fig. 1 (Wegner form). SW: equal-weight means of the gas and
# liquid parts of the four-run mean ln Q.
const TC_SW = Dict(λ => (ss = sort(filter(x -> x.λ == λ, SWS), by = x -> x.T_star);
                         estimate_critical_point(getfield.(ss, :T_star), [x.coex.Nliq / V_SW for x in ss],
                                                 [x.coex.Ngas / V_SW for x in ss]).Tc) for λ in LAMS)
const TC_RED = Dict(0.0 => CP.Tc / ε_kB_Ar, (λ => TC_SW[λ] for λ in LAMS)...)
@printf("T_c (Wegner fit): LJ %.2f K (T* = %.4f)%s\n", CP.Tc, TC_RED[0.0],
        join([@sprintf(", SW λ = %.2f: %.4f", λ, TC_SW[λ]) for λ in LAMS]))
"Measure, predicted and observed circle reach for one row of the Lee files (Vs = box volume of that system)."
function reach(r, Vs)
    L = r.L; ΔN, ΔVar = Vs * L.Δ[1], Vs * L.Δ[2]
    e = r.err[:, 1]             # first order, on the four-run mean ln Q
    i = findfirst(>=(0.5), e)   # first miss of ≥ ½ spacing
    # k_x: zero index at which the third-order term of Lee's series overtakes the second-order one,
    # |b3 w| = |b2| with |w_k| = 2π(k+½)/V, i.e. k_x + ½ = 3ΔN²ΔVar / (2π |3ΔVar² − ΔN ΔK3|)
    kx = Vs * abs(L.b[2] / L.b[3]) / (2π) - 0.5
    (λ = r.λ, T = r.T, Tred = r.T / TC_RED[r.λ], meas = ΔN / sqrt(abs(ΔVar)),
     kpred = ΔN / sqrt(2π * abs(ΔVar)) - 0.5, kobs = i === nothing ? missing : i - 1, kx = kx)
end
REACH = vcat([reach(r, V_LJ) for r in ljrows], [reach(r, V_SW) for r in swrows])

q = [(x.kobs + 0.5) / (x.kpred + 0.5) for x in REACH if x.kobs !== missing]
@printf("circle reach, %d state points: observed/predicted (k+½) = %.2f–%.2f, median %.2f\n", length(q), extrema(q)..., median(q))
println(" system          T*     T/T_c   ΔN/√|ΔVar|   k1 pred   k1 obs    k_x")
for x in REACH
    @printf(" %-14s  %.3f   %.3f     %6.1f      %5.1f     %-4s  %6.1f\n", x.λ == 0 ? "LJ" : @sprintf("SW λ=%.2f", x.λ),
            x.T, x.Tred, x.meas, x.kpred, x.kobs === missing ? "–" : string(x.kobs), x.kx)
end

# (a) the reach estimate against the ratio of the third- to the second-order term at the predicted reach,
# |b3 w| / |b2| at k1 = (k1 + ½)/(k_x + ½): above 1 the series terms grow with order before k1
SYSM_R = Dict(0.0 => :circle, 1.5 => :diamond, 1.75 => :utriangle)
SYSL_R = Dict(0.0 => "Lennard–Jones", 1.5 => "SW λ = 1.50", 1.75 => "SW λ = 1.75")
TRLIM  = extrema(getfield.(REACH, :Tred))
pe = plot(xscale = :log10, xlims = (0.08, 25), ylims = (0.6, 1.5), legend = :topleft,
          xlabel = "3rd- / 2nd-order term at predicted reach",
          ylabel = "observed / predicted reach", title = "(a) where the circle's reach is predicted")
vline!(pe, [1.0], lc = INK2, ls = :dash, lw = 1, label = "")
hline!(pe, [1.0], lc = INK2, lw = 1, label = "")
for λ in (0.0, SW_LAMBDAS...)
    s = [x for x in REACH if x.λ == λ && x.kobs !== missing]
    scatter!(pe, [(x.kpred + 0.5) / (x.kx + 0.5) for x in s], [(x.kobs + 0.5) / (x.kpred + 0.5) for x in s], m = SYSM_R[λ],
             ms = 6, msw = 0.5, msc = INK2, marker_z = getfield.(s, :Tred), color = cgrad(:coolwarm), clims = TRLIM,
             colorbar = true, colorbar_title = "\$T/T_c\$", label = "")
    scatter!(pe, [NaN], [NaN], m = SYSM_R[λ], ms = 6, mc = :white, msw = 1, msc = INK2, label = SYSL_R[λ])
end
qa = [(x.kobs + 0.5) / (x.kpred + 0.5) for x in REACH if x.kobs !== missing && x.kx > x.kpred]
qb = [(x.kobs + 0.5) / (x.kpred + 0.5) for x in REACH if x.kobs !== missing && x.kx <= x.kpred]
@printf("k_x > k1 pred: %d points, obs/pred %.2f–%.2f;  k_x < k1 pred: %d points, obs/pred %.2f–%.2f\n",
        length(qa), extrema(qa)..., length(qb), extrema(qb)...)

# (b) share of |2nd| + |3rd| carried by each order at zero k = K_BAR, every LJ isotherm; the third order overtakes
# the second where its share crosses ½
const K_BAR = 10
term_size(L, k, n) = abs(L.b[n]) * (2π * (k + 0.5) / V_LJ)^n / (2π / (V_LJ * L.Δ[1]))
ljsorted = sort(ljrows, by = r -> r.T)
t2b = [term_size(r.L, K_BAR, 2) for r in ljsorted]; t3b = [term_size(r.L, K_BAR, 3) for r in ljsorted]
Tb  = [r.T for r in ljsorted]
f3 = t3b ./ (t2b .+ t3b)
pT = plot(ylims = (0, 1.02), xlabel = "\$T^*\$", ylabel = "share of correction", legend = :left,
          xticks = 0.75:0.05:1.20, xrotation = 45, yticks = 0:0.25:1,
          title = @sprintf("(b) second- vs third-order term, zero \$k = %d\$", K_BAR))
bar!(pT, Tb, ones(length(Tb)), bar_width = 0.035, fc = "#c0392b", lw = 0, label = "3rd-order term")
bar!(pT, Tb, 1 .- f3,          bar_width = 0.035, fc = ORANGE,    lw = 0, label = "2nd-order term")
hline!(pT, [0.5], lc = INK2, ls = :dash, lw = 1, label = "")
println(" T*     |2nd| at k=$K_BAR   |3rd| at k=$K_BAR   3rd/2nd   3rd share")
for (T, a, b, f) in zip(Tb, t2b, t3b, f3); @printf(" %.2f   %.4f        %.4f        %.3f     %.2f\n", T, a, b, b / a, f); end

# (c), (d) density from all N_max zeros predicted by Lee's expansion at order n (k = 0 … N_max/2 − 1 above the axis
# plus conjugates; indices past the ring wrap beyond θ = π; no computed zeros used), ρ = (1/V) Σ z/(z − z_k), vs ln Q
lnQ_Nvar(s, t) = (g = s.lnQ .+ s.N .* (s.coex.u + t); p = exp.(g .- lse(g)); m = sum(p .* s.N); (m, sum(p .* (s.N .- m) .^ 2)))
tv = range(-0.2, 0.2, length = 801)
function nmax_lee_zeros(s, n)
    r = lee_predictions(s; nz = length(s.N) ÷ 2)
    p = filter(isfinite, getfield(r, (:z1, :z2, :z3)[n]))
    return vcat(p, conj.(p))
end
rho_lee(s, n) = (Z = nmax_lee_zeros(s, n); [real(sum(z ./ (z .- Z))) / s.V for z in exp.(s.coex.u .+ tv)])
function nmax_rho_panel(s, tag)
    p = plot(xlabel = "\$\\beta\\mu - \\beta\\mu_c\$", ylabel = "\$\\rho\\;(\\sigma^{-3})\$", legend = :topleft,
             title = @sprintf("(%s) \$\\rho\$ from Lee's zeros, \$T^* = %.2f\$", tag, s.T_star))
    plot!(p, tv, [lnQ_Nvar(s, t)[1] / s.V for t in tv], lc = :black, lw = 2.6, label = "from ln Q")
    for (n, col, ls) in ((1, BLUE, :dot), (2, ORANGE, :dash), (3, "#c0392b", :dashdot))
        plot!(p, tv, rho_lee(s, n), lc = col, lw = 2, ls = ls, label = "order $n")
    end
    return p
end
pR1 = nmax_rho_panel(LJ["87.79"], "c"); pR2 = nmax_rho_panel(LJ["140.46"], "d")
println(" T*     median |ρ_n − ρ_lnQ| over |βμ − βμ_c| ≤ 0.2 (σ⁻³), orders 1 / 2 / 3")
for s in (LJ["87.79"], LJ["140.46"])
    ρx = [lnQ_Nvar(s, t)[1] / s.V for t in tv]
    @printf(" %.2f   %s\n", s.T_star, join([@sprintf("%.2g", median(abs.(rho_lee(s, n) .- ρx))) for n in 1:3], " / "))
end

fig8 = plot(pe, pT, pR1, pR2, layout = (2, 2), size = (2100, 1500), dpi = 200,
            left_margin = 10Plots.mm, bottom_margin = 10Plots.mm)
save_fig(fig8, "fig8_series_breakdown")

## Fig. 4: the square-well fluid

The same analysis for $\lambda = 1.5$ and $1.75$. In the paper's Fig. 4 (drawn by `make_figures_mpl.ipynb`),
(a) and (b) are the zero maps and (c) is the saturation pressure, which `scripts/sw_psat.jl` (last cell) adds to
`data/figure_data.jld2`. The Plots.jl preview below differs in (c): it shows the coexistence fugacity $\ln z_c$
against NIST instead.

In [ ]:
# (a), (b): all isotherms at each range overlaid on the full plane in raw fugacity, as Fig. 2
function sw_overlay_panel(λ, tag)
    ss = sort(filter(x -> x.λ == λ, SWS), by = x -> x.T_star)
    Ts = [x.T_star for x in ss]; zmax = maximum(x.coex.z for x in ss)
    p = plot(xlabel = "\$\\mathrm{Re}\\,z\$", ylabel = "\$\\mathrm{Im}\\,z\$", aspect_ratio = :equal, legend = :outerright,
             legend_title = "\$T^*\$", xticks = [-0.04, 0, 0.04], yticks = [-0.04, 0, 0.04], xlims = (-1.3zmax, 1.3zmax), ylims = (-1.3zmax, 1.3zmax),
             title = @sprintf("(%s) \$\\lambda = %.2f\$, all isotherms", tag, λ))
    for x in ss
        col = TGRAD[(x.T_star - Ts[1]) / (Ts[end] - Ts[1])]
        scatter!(p, real.(x.z), imag.(x.z), mc = col, ms = 1.8, msw = 0, alpha = 0.9, label = @sprintf("%.2f", x.T_star))
    end
    return p
end
maps = [sw_overlay_panel(1.5, "a"), sw_overlay_panel(1.75, "b")]
pd = plot(title = "(c) coexistence fugacity (Λ = 1, so y = z)", xlabel = "\$T^*\$", ylabel = "\$\\ln z_c\$", legend = (0.3, 0.25),
          xticks = ([1.0, 1.2, 1.4, 1.6], ["1.0", "1.2", "1.4", "1.6"]),
          right_margin = 6Plots.mm)
# NIST SAT-TMMC, λ = 1.5, V = 729σ³ (same box); their ln z = βμ with Λ = σ, the same quantity as ours.
# https://www.nist.gov/mml/csd/chemical-informatics-group/sat-tmmc-liquid-vapor-coexistence-properties-square-well-fluid
const NIST_SW15_T   = [0.95, 1.00, 1.05, 1.10, 1.15]
const NIST_SW15_LNZ = [-4.149, -3.865, -3.614, -3.391, -3.194]
scatter!(pd, NIST_SW15_T, NIST_SW15_LNZ, m = :square, mc = :white, msc = INK2, msw = 1.2, ms = 5,
         label = "NIST, \$\\lambda = 1.50\$")
for λ in LAMS
    ss = sort(filter(x -> x.λ == λ, SWS), by = x -> x.T_star)
    plot!(pd, [x.T_star for x in ss], [x.coex.u for x in ss], yerror = [x.lnz_sd for x in ss], lc = swcol(λ), lw = 2,
          m = :circle, mc = swcol(λ), ms = 4, msw = 0.5, label = @sprintf("\$\\lambda = %.2f\$", λ))
end
fig4 = plot(maps..., pd, layout = (1, 3), size = (2200, 680), dpi = 200, left_margin = 8Plots.mm, bottom_margin = 9Plots.mm)
save_fig(fig4, "fig4_square_well")

## Numbers quoted in the text

The quantities the text quotes, from the same objects as the figures, written to `cache/numbers_for_manuscript.csv`
and `cache/numbers_square_well.csv`.

In [ ]:
reach_of(λ, T) = REACH[findfirst(x -> x.λ == λ && abs(x.T - T) < 1e-3, REACH)]
open(joinpath(CACHE, "numbers_for_manuscript.csv"), "w") do io
    println(io, "T_K,T_star,z_coex,ln_z_coex,DeltaN,N_b,ring,inner,outer,Re_z1,Im_z1,bend_pred,sep_over_mismatch,k1_reach_pred,k1_reach_obs,k_x_third_over_second")
    for (s, z1) in zip(SYS, z1s)
        B = BEND[s.name]; R = reach_of(0.0, s.T_star)
        println(io, join((s.T_K, s.T_star, s.coex.z, s.coex.u, s.coex.ΔN, s.coex.Nb, count(==(:ring), s.cls),
                          count(==(:inner), s.cls), count(==(:outer), s.cls), real(z1), imag(z1),
                          B, R.meas, R.kpred, coalesce(R.kobs, -1), R.kx), ","))
    end
end
open(joinpath(CACHE, "numbers_square_well.csv"), "w") do io
    println(io, "lambda,T_star,N_max,ln_z_coex,ln_z_sd,DeltaN,ring,inner,outer,bend_pred,sep_over_mismatch,k1_reach_pred,k1_reach_obs,k_x_third_over_second")
    for s in SWS
        B = BEND[s.name]; R = reach_of(s.λ, s.T_star)
        println(io, join((s.λ, s.T_star, last(s.N), s.coex.u, s.lnz_sd, s.coex.ΔN, count(==(:ring), s.cls),
                          count(==(:inner), s.cls), count(==(:outer), s.cls), B, R.meas, R.kpred, coalesce(R.kobs, -1), R.kx), ","))
    end
end
println(" T*    ln z_c       ΔN      N_b  ring  inner outer  Im z1/z_c      ΔVar/ΔN   reach pred  obs")
for (s, z1) in zip(SYS, z1s)
    B = BEND[s.name]; R = reach_of(0.0, s.T_star)
    @printf(" %.2f  %.6f  %6.1f   %3d  %3d   %3d   %3d   %.4e     %.4f    %5.1f   %s\n", s.T_star, s.coex.u,
            s.coex.ΔN, s.coex.Nb, count(==(:ring), s.cls), count(==(:inner), s.cls), count(==(:outer), s.cls),
            imag(z1) / s.coex.z, B, R.kpred, R.kobs === missing ? "–" : string(R.kobs))
end
println("\nwrote cache/numbers_for_manuscript.csv, cache/numbers_square_well.csv")

In [ ]:
# ── export: the arrays each paper figure plots, for the matplotlib notebook make_figures_mpl.ipynb ──
# One HDF5 file (JLD2 writes HDF5); complex numbers split into re/im, classes coded 0 inside, 1 ring, 2 outside.
const EXPORT = joinpath(FIGOUT, "data", "figure_data.jld2"); mkpath(dirname(EXPORT))
clscode(c) = c === :inner ? 0 : c === :ring ? 1 : 2
jldopen(EXPORT, "w") do f
    put(k, z::AbstractVector{<:Complex}) = (f[k * "_re"] = Float64.(real.(z)); f[k * "_im"] = Float64.(imag.(z)))
    put(k, x) = (f[k] = x)
    # Fig. 1
    for s in SYS
        g = s.lnQ .+ s.N .* s.coex.u
        put("fig1/T$(s.T_str)/N", Float64.(s.N)); put("fig1/T$(s.T_str)/lnQ", s.lnQ)
        put("fig1/T$(s.T_str)/PN", exp.(g .- lse(g))); put("fig1/T$(s.T_str)/Tstar", s.T_star)
    end
    put("fig1/T_K", TEMP_K)
    put("fig1/rho_liq", getfield.(TAB, :ρl)); put("fig1/rho_liq_sd", getfield.(TAB, :ρl_sd))
    put("fig1/rho_vap", getfield.(TAB, :ρv)); put("fig1/rho_vap_sd", getfield.(TAB, :ρv_sd))
    put("fig1/tmmc_liq", TMMC_ρLIQ); put("fig1/tmmc_vap", TMMC_ρVAP)
    put("fig1/exp_liq", EXP_ρLIQ); put("fig1/exp_vap", EXP_ρVAP)
    put("fig1/Tc", CP.Tc); put("fig1/rhoc", CP.ρc); put("fig1/exp_Tc", EXP_TC); put("fig1/exp_rhoc", EXP_ρC)
    put("temps", collect(TEMPS))
    # Figs. 2, 3: every LJ zero, z_c, class
    for s in SYS
        put("lj/T$(s.T_str)/z", s.z); put("lj/T$(s.T_str)/zc", s.coex.z); put("lj/T$(s.T_str)/Tstar", s.T_star)
        put("lj/T$(s.T_str)/cls", clscode.(s.cls))
    end
    # Fig. 4: SW zeros, ln z_c with run sd, NIST
    for x in SWS
        k = @sprintf("sw/lam%.2f/T%.3f", x.λ, x.T_star)
        put(k * "/z", x.z); put(k * "/lnzc", x.coex.u); put(k * "/lnzc_sd", x.lnz_sd); put(k * "/Tstar", x.T_star)
    end
    put("sw/nist_T", NIST_SW15_T); put("sw/nist_lnz", NIST_SW15_LNZ)
    # Fig. 5: four runs and the averaged ln Q at FIG5_T, in units of z_c
    put("fig5/avg", s3.z ./ zc3); put("fig5/Tstar", s3.T_star)
    for r in 1:4; put("fig5/run$r", RR[r] ./ zc3); end
    # Fig. 6: ρ and κ_T from ln Q (lines) and from the roots (points)
    put("fig6/t", βμ .- s3.coex.u); put("fig6/rho", ρ_direct); put("fig6/kappa", κ_direct)
    put("fig6/tm", βμm .- s3.coex.u); put("fig6/rho_roots", ρ_roots); put("fig6/kappa_roots", κ_roots)
    # Fig. 7: exact zeros and Lee's predictions (whole ring, and the first NZOOM), in units of z_c
    for T in ("87.79", "134.61", "140.46")
        s = LJ[T]; nring = count(i -> s.cls[i] === :ring && imag(s.z[i]) > 0, eachindex(s.z))
        put("fig7/T$T/w", s.z ./ s.coex.z); put("fig7/T$T/Tstar", s.T_star)
        for (lab, nz) in (("full", nring), ("zoom", NZOOM))
            r = lee_predictions(s; nz = nz)
            for n in 1:3; put("fig7/T$T/$lab/z$n", getfield(r, (:z1, :z2, :z3)[n]) ./ r.zc); end
            lab == "zoom" && put("fig7/T$T/zoom/exact", r.exact[1:min(NZOOM, length(r.exact))] ./ r.zc)
        end
    end
    # Fig. 8: reach, term shares, ρ from all N_max Lee zeros
    put("fig8/lam", getfield.(REACH, :λ)); put("fig8/Tred", getfield.(REACH, :Tred))
    put("fig8/kobs", [x.kobs === missing ? NaN : Float64(x.kobs) for x in REACH])
    put("fig8/kpred", getfield.(REACH, :kpred)); put("fig8/kx", getfield.(REACH, :kx))
    put("fig8/Tb", Tb); put("fig8/t2", t2b); put("fig8/t3", t3b); put("fig8/kbar", K_BAR)
    put("fig8/t", collect(tv))
    for T in ("87.79", "140.46")
        s = LJ[T]
        put("fig8/T$T/rho", [lnQ_Nvar(s, t)[1] / s.V for t in tv]); put("fig8/T$T/Tstar", s.T_star)
        for n in 1:3
            put("fig8/T$T/rho$n", rho_lee(s, n))
        end
    end
end
println("wrote ", EXPORT)

In [ ]:
# ── Fig. 4(c) saturation pressures: computed from the SW ln Q and appended to figure_data.jld2 by a standalone
# script, so run it after the export cell above (which rewrites the file).
run(`julia $(joinpath(FIGOUT, "scripts", "sw_psat.jl"))`)